# 05 - MiniLM Embeddings

## Objective

Generate embeddings for Pipeline 1 using:

- Recursive Character Chunking
- all-MiniLM-L6-v2

Pipeline:

Documents
→ Recursive Chunking
→ MiniLM Embeddings
→ ChromaDB
→ Qwen3:14B

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## Import Project Modules

Load configuration and document loader components required for the embedding pipeline.

In [2]:
from src.config import DOCS_PATH
from src.loaders.document_loader import DocumentLoader

loader = DocumentLoader(DOCS_PATH)

documents = loader.load()

print(
    f"Source Pages: {len(documents):,}"
)

Starting Document Loading...

Loading PDF : iSupplier_portal.pdf

Loading PDF : Purchasing_manual.pdf


DOCUMENT LOADING SUMMARY
PDF Files Loaded   : 2
DOCX Files Loaded  : 0
XLSX Files Loaded  : 0
Total Documents    : 1502
Source Pages: 1,502


In [3]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

## Create Recursive Chunks

Pipeline 1 uses Recursive Character Chunking with:

- Chunk Size: 1000
- Chunk Overlap: 200

This strategy favors retrieval precision.

In [4]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

recursive_chunks = (
    recursive_splitter.split_documents(
        documents
    )
)

print(
    f"Chunks Created: {len(recursive_chunks):,}"
)

Chunks Created: 3,519


In [5]:
import torch

## Verify GPU Availability

Detect available hardware and determine whether embeddings will be generated using CPU or GPU.

In [6]:
print("=" * 80)

print("DEVICE INFORMATION")

print("=" * 80)

print(
    f"CUDA Available : {torch.cuda.is_available()}"
)

if torch.cuda.is_available():

    print(
        f"GPU : {torch.cuda.get_device_name(0)}"
    )

DEVICE INFORMATION
CUDA Available : True
GPU : NVIDIA RTX A5000 Laptop GPU


In [7]:
from langchain_huggingface import (
    HuggingFaceEmbeddings
)

In [8]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    f"Embedding Device: {device}"
)

Embedding Device: cuda


## Initialize MiniLM Embedding Model

Load the all-MiniLM-L6-v2 model used by Pipeline 1.

This model generates 384-dimensional vectors and is optimized for fast semantic retrieval.

In [9]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={
        "device": device
    }
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Validate Embeddings

Generate a sample embedding and verify vector dimensions.

In [10]:
sample_embedding = embedding_model.embed_query(
    "What is Oracle Purchasing?"
)

In [11]:
print("=" * 80)

print("EMBEDDING VALIDATION")

print("=" * 80)

print(
    f"Embedding Dimension: {len(sample_embedding)}"
)

EMBEDDING VALIDATION
Embedding Dimension: 384


In [12]:
print(
    sample_embedding[:10]
)

[-0.0576089471578598, 0.0245400071144104, -0.07278977334499359, -0.042161598801612854, -0.058800458908081055, 0.007792038377374411, 0.06583680212497711, 0.06265843659639359, -0.047528937458992004, 0.057991549372673035]


In [13]:
sample_chunk = recursive_chunks[0]

In [14]:
sample_vector = embedding_model.embed_query(
    sample_chunk.page_content
)

In [15]:
print("=" * 80)

print("CHUNK EMBEDDING CHECK")

print("=" * 80)

print(
    f"Vector Length: {len(sample_vector)}"
)

CHUNK EMBEDDING CHECK
Vector Length: 384


In [16]:
print("=" * 80)

print("CHUNK PREVIEW")

print("=" * 80)

print(
    sample_chunk.page_content[:1000]
)

CHUNK PREVIEW
Oracle® iSupplier Portal
 User's Guide 
Release 12.2
 Part No. E48972-13
October 2025


In [17]:
embedding_info = {
    "Model":
        "all-MiniLM-L6-v2",

    "Dimensions":
        len(sample_embedding),

    "Chunk Count":
        len(recursive_chunks),

    "Device":
        device
}

In [18]:
import pandas as pd

pd.DataFrame(
    embedding_info.items(),
    columns=[
        "Property",
        "Value"
    ]
)

,Property,Value
0,Model,all-MiniLM-L6-v2
1,Dimensions,384
2,Chunk Count,3519
3,Device,cuda


## Export Embedding Metadata

Persist notebook diagnostics for future comparison and GitHub documentation.

In [19]:
pd.DataFrame(
    embedding_info.items(),
    columns=[
        "Property",
        "Value"
    ]
).to_csv(
    "../screenshots/embeddings/minilm_embedding_info.csv",
    index=False
)

print(
    "Embedding metadata exported."
)

Embedding metadata exported.


# Conclusion

## MiniLM Embeddings

Model:

- all-MiniLM-L6-v2

Characteristics:

- Lightweight
- Fast
- 384-dimensional vectors
- Suitable for large-scale retrieval

## Pipeline

Recursive Chunking
→ MiniLM Embeddings
→ ChromaDB
→ Qwen3:14B

## Next Step

Proceed to:

06_BGEM3_Embeddings.ipynb